# Module 8: Advanced Topics — Token Swarms, Extensions & Open Problems

**Estimated time: 30 minutes**

## 8.1 Token Swarms: Beyond Shared Architectures

A fundamental limitation of weight-space swarms: **all models must share the same architecture.** Token Swarms solve this by moving the search from weight space to **output probability space**.

Instead of each particle being a set of model weights, each particle is a **weighted combination of output probability distributions**:

$$p_{combined}(v|x) = \sum_j w_{i,j} \cdot p_j(v|x)$$

| Aspect | Weight Swarms | Token Swarms |
|--------|--------------|-------------|
| Architecture requirement | All same | Any architecture |
| What's searched | LoRA weight space (~18M dims) | Probability mixture space (M dims) |
| Can create new behaviors | Yes (weight synergies) | Limited (convex combinations) |

Preliminary results: Token Swarms with 4 Gemma + 4 Mistral models achieves **+29.3% average improvement**.

In [ ]:
import numpy as np

class TokenSwarmParticle:
    """A particle in Token Swarms — its position is a set of mixture weights over models."""
    
    def __init__(self, n_models):
        # Position: mixture weights over models (on the simplex)
        self.weights = np.random.dirichlet(np.ones(n_models))
        self.velocity = np.random.randn(n_models) * 0.1
        self.personal_best_weights = self.weights.copy()
        self.personal_best_score = -np.inf
    
    def combine_predictions(self, model_probs):
        """
        Combine model output probabilities using this particle's weights.
        
        Args:
            model_probs: list of probability distributions, one per model
                         each of shape (vocab_size,)
        Returns:
            combined probability distribution
        """
        combined = sum(w * p for w, p in zip(self.weights, model_probs))
        return combined / combined.sum()  # renormalize

# Demonstrate with toy example
n_models = 4
vocab_size = 10

# Simulated model outputs (probabilities over 10 tokens)
np.random.seed(42)
model_probs = [np.random.dirichlet(np.ones(vocab_size) * 0.5) for _ in range(n_models)]

# Create a particle and combine
particle = TokenSwarmParticle(n_models)
combined = particle.combine_predictions(model_probs)

print(f"Particle weights: {particle.weights.round(3)}")
print(f"\nIndividual model predictions (argmax token):")
for i, probs in enumerate(model_probs):
    print(f"  Model {i}: token {np.argmax(probs)} (prob={probs.max():.3f})")
print(f"\nCombined prediction: token {np.argmax(combined)} (prob={combined.max():.3f})")

## 8.2 Limitations and Honest Assessment

### Limitation 1: Adaptation, Not Learning
Model Swarms reallocates existing knowledge — it doesn't create new knowledge. Using perplexity as utility doesn't help.

### Limitation 2: Local Optima
PSO provides no convergence guarantee. Results are stochastic — different seeds produce different outcomes.

### Limitation 3: Evaluation Cost
Each iteration requires N model evaluations. Scales poorly with model size (70B = ~10x more compute than 7B per evaluation).

### Limitation 4: Utility Function Design
The algorithm is only as good as its utility function. Biased data → biased models. Too few examples → overfitting.

### Limitation 5: LoRA Constraint
All experts must be LoRA adapters of the same base model, limiting the expert pool.

## 8.3 Extensions and Future Directions

### Extension 1: Adaptive Hyperparameters
Increase inertia when diversity drops, adjust step length based on improvement rate.

### Extension 2: Multi-Objective Optimization (MOPSO)
Maintain a **Pareto front** of non-dominated solutions instead of a single global best. Gives users a menu of trade-off models.

### Extension 3: Continual Adaptation
Start from previous search's best when new data arrives. Add/remove experts over time.

### Extension 4: Heterogeneous Search Spaces
Combine weight-space search (same architecture) with token-space search (cross-architecture).

### Extension 5: Learned Utility Functions
```
human ranks outputs → train reward model → use as utility → run swarm
```

## 8.4 The Broader Landscape

```
Simple ←───────────────────────────────────────→ Complex
Averaging    Merging     Routing      Swarms      MoE Training

- Uniform    - TIES      - cBTM       - Model     - GShard
  Soup       - DARE      - Pack of      Swarms    - Switch
- Greedy     - SLERP       LLMs       - Token       Transformer
  Soup       - Model     - Branch-      Swarms    - Mixtral
               Stocks      Train-Merge
```

The trend is toward methods that are more dynamic, data-efficient, modular, and theoretically grounded. Model Swarms advances the first three but lacks the fourth.

## 8.5 Ethical Considerations

The flexibility of the utility function is both a strength and a risk:
- **Toxicity optimization**: A toxicity score as utility would produce maximally toxic models
- **Bias amplification**: Biased validation data → amplified biases
- **Reward hacking**: Exploitable shortcuts will be found

Mitigations: careful utility design, red-teaming, restricting the expert pool to aligned models.

## 8.6 Course Conclusion

You've completed a deep dive into Model Swarms:

| Module | Topic |
|--------|-------|
| 1 | The problem and why existing approaches fall short |
| 2 | Swarm intelligence and PSO — the optimization foundation |
| 3 | LoRA adapters and merging — the representation foundation |
| 4 | The complete algorithm — the core contribution |
| 5 | The implementation — theory becomes code |
| 6 | Hands-on experimentation |
| 7 | Results analysis and ablations |
| 8 | Extensions, limitations, and open problems |

### Key Takeaways

1. **Simple algorithms can be powerful** when applied to the right representation. PSO is 30 years old. LoRA is straightforward. Their combination produces SOTA results.

2. **Diversity is a resource.** Effectiveness depends critically on having diverse initial experts.

3. **Emergence is real.** Combining weights can produce capabilities absent from any individual model.

4. **Utility function design is the new feature engineering.** The bottleneck shifts to specifying *what* to optimize.

5. **The field is moving fast.** These techniques are the foundation, but methods will continue to advance.

## Exercise 8.1: Token Swarms Extension

Extend the `TokenSwarmParticle` class to implement a full search:

In [ ]:
class TokenSwarm:
    def __init__(self, n_models, n_particles=10):
        self.n_models = n_models
        self.particles = [TokenSwarmParticle(n_models) for _ in range(n_particles)]
        self.global_best_weights = None
        self.global_best_score = -np.inf
    
    def evaluate_particle(self, particle, model_probs_dataset, labels):
        """
        Evaluate a particle on a dataset.
        
        Args:
            particle: TokenSwarmParticle
            model_probs_dataset: list of (list of model probs), one per example
            labels: correct token indices
        Returns:
            accuracy score
        """
        correct = 0
        for model_probs, label in zip(model_probs_dataset, labels):
            combined = particle.combine_predictions(model_probs)
            if np.argmax(combined) == label:
                correct += 1
        return correct / len(labels)
    
    def step(self, model_probs_dataset, labels, 
             inertia=0.2, cognitive=0.3, social=0.4):
        """Run one PSO iteration on the mixture weight space."""
        # TODO: Implement PSO update on particle.weights
        # Remember: weights should stay on the simplex (non-negative, sum to 1)
        # Hint: update weights, then project back to simplex via normalization
        pass

# TODO: Create a toy dataset and test the search
# Simulated: 50 examples, 4 models, vocab of 5 tokens

## Exercise 8.2: Multi-Objective Extension

Sketch (in pseudocode or Python) how you would modify Model Swarms to maintain a Pareto front:
1. Define dominance for two particles with scores on two tasks
2. Implement Pareto front maintenance
3. Modify the social term to attract toward a random Pareto-front point

*Your design:*



## Exercise 8.3: Research Proposal

Write a 1-page research proposal for a follow-up paper:
1. **Problem statement**: What limitation do you address?
2. **Proposed method**: What's your approach?
3. **Expected results**: What do you predict?
4. **Evaluation plan**: How will you measure success?
5. **Baselines**: What do you compare against?

*Your proposal:*



## Exercise 8.4: Literature Comparison

Read the abstract of ONE related paper:
- [Model Soups](https://arxiv.org/abs/2203.05482) (Wortsman et al., 2022)
- [TIES-Merging](https://arxiv.org/abs/2306.01708) (Yadav et al., 2023)
- [Evolutionary Model Merge](https://arxiv.org/abs/2403.13187) (Akiba et al., 2024)

Write a 2-paragraph comparison with Model Swarms.

*Your comparison:*



---

**Congratulations on completing the course!**

**[Back to Course Overview](README.md)** | **[Coding Challenges](exercises.ipynb)**